# Thesis Experiment

## Data Cleaning

In [3]:
import pandas as pd
import numpy as np
import os
import glob

### BEA I/O Tables

In [6]:
print("Processing Annual BEA Input-Output Tables...")

# 1. The Expanded BEA-to-GICS Dictionary (Optimized for the BEA 'Name' column)
bea_to_gics = {
    'Technology': ['Computer and electronic', 'Data processing', 'Computer systems design', 'Information', 'Software'],
    'Health Care': ['Ambulatory health', 'Hospitals', 'Nursing', 'Pharmaceutical', 'Medical'],
    'Energy': ['Oil and gas', 'Petroleum', 'Pipeline'],
    'Financials': ['Federal Reserve', 'Credit intermediation', 'Securities', 'Insurance', 'Funds, trusts', 'Financial', 'Rental and leasing'],
    'Industrials': ['Machinery', 'Electrical equipment', 'Aerospace', 'Rail transportation', 'Air transportation', 
                    'Fabricated metal', 'Other transportation equipment', 'Truck transportation', 'Water transportation',
                    'Transit and ground passenger', 'Warehousing and storage', 'Waste management', 'Construction', 
                    'Management of companies', 'Administrative and support'],
    'Consumer Staples': ['Food and beverage', 'Farms', 'Paper products', 'Grocery'],
    'Consumer Discretionary': ['Motor vehicles', 'Apparel', 'Accommodation', 'Food services', 'Arts, entertainment', 
                               'Amusements, gambling', 'Retail', 'Textile', 'Furniture', 'Motor vehicle and parts dealers',
                               'General merchandise stores', 'Performing arts, spectator sports'],
    'Utilities': ['Utilities'],
    'Materials': ['Mining', 'Chemical products', 'Primary metals', 'Plastics and rubber', 'Nonmetallic mineral', 'Wood products'],
    'Real Estate': ['Real estate', 'Housing'],
    'Communication': ['Publishing', 'Motion picture', 'Broadcasting', 'Telecommunications']
}

def get_gics_sector(bea_name):
    bea_name_str = str(bea_name).lower()
    for gics, bea_list in bea_to_gics.items():
        if any(str(bea_item).lower() in bea_name_str for bea_item in bea_list):
            return gics
    return 'Other'

annual_weights = {}

# 2. Loop through the years
for year in range(2004, 2025):
    file_path = f'./Data/IO_Tables/{year}.csv'
    
    if not os.path.exists(file_path):
        continue
        
    # Skip the 4 rows of BEA metadata at the top of the CSV
    df_io = pd.read_csv(file_path, skiprows=4)
    
    # Drop empty rows (like footer notes where the Code column is NaN)
    df_io.dropna(subset=[df_io.columns[0]], inplace=True)
    
    # Dynamically define the code and name columns based on the file's structure
    code_col = df_io.columns[0]
    name_col = df_io.columns[1]
    
    # Map the Input Sectors (Rows) using the plain-English Name column
    df_io['Input_Sector'] = df_io[name_col].apply(get_gics_sector)
    
    # Identify the columns that contain the Consuming Industries (everything after Code, Name, and our new Input_Sector)
    consuming_cols = [col for col in df_io.columns if col not in [code_col, name_col, 'Input_Sector']]
    
    # Melt the dataframe (CRITICAL FIX: Keeping name_col in id_vars)
    melted = df_io.melt(id_vars=[code_col, name_col, 'Input_Sector'], 
                        value_vars=consuming_cols,
                        var_name='Consuming_Industry', 
                        value_name='Dollar_Value')
    
    # Clean the dollar values (remove commas, convert to float)
    melted['Dollar_Value'] = pd.to_numeric(melted['Dollar_Value'].astype(str).str.replace(',', ''), errors='coerce').fillna(0)
    
    # Map the Consuming Industries (Columns) to GICS
    melted['Output_Sector'] = melted['Consuming_Industry'].apply(get_gics_sector)
    
    # Filter out 'Other' (Non-GICS inputs like taxes, labor, or government)
    gics_only = melted[(melted['Input_Sector'] != 'Other') & (melted['Output_Sector'] != 'Other')]
    
    # Aggregate dollars into the 11x11 matrix
    matrix_11x11 = gics_only.groupby(['Input_Sector', 'Output_Sector'])['Dollar_Value'].sum().unstack(fill_value=0)
    
    # Remove the diagonal (j=k) because systemic risk propagates downstream (a sector cannot inherit network fragility from itself)
    np.fill_diagonal(matrix_11x11.values, 0)
    
    # Calculate ω_jk: The proportion of Sector J's inputs sourced from Sector K
    column_sums = matrix_11x11.sum(axis=0)
    omega_matrix = matrix_11x11.div(column_sums.replace(0, np.nan), axis=1).fillna(0)
    
    # Store the final weight matrix for this year
    annual_weights[year] = omega_matrix

print(f"Successfully calculated network weights for {len(annual_weights)} years.")

Processing Annual BEA Input-Output Tables...
Successfully calculated network weights for 21 years.


In [8]:
import pickle

# Save the annual_weights dict for downstream visualization
with open('./Data/annual_weights.pkl', 'wb') as f:
    pickle.dump(annual_weights, f)

# Also save the most recent year as a standalone CSV for sanity-checking
latest_year = max(annual_weights.keys())
annual_weights[latest_year].to_csv(f'./Data/omega_matrix_{latest_year}.csv')

print(f"Saved annual_weights.pkl ({len(annual_weights)} years)")
print(f"Saved omega_matrix_{latest_year}.csv (most recent year)")

Saved annual_weights.pkl (21 years)
Saved omega_matrix_2024.csv (most recent year)


In [10]:
print("Calculating dynamic Supplier HHI...")

# 1. Load all 11 of your monthly HHI CSVs into one master DataFrame
hhi_files = glob.glob('./Data/HHI/*.csv')
hhi_list = []
for file in hhi_files:
    temp_df = pd.read_csv(file)
    hhi_list.append(temp_df)

master_hhi = pd.concat(hhi_list, ignore_index=True)
master_hhi['Date'] = pd.to_datetime(master_hhi['Date'])
master_hhi['Year'] = master_hhi['Date'].dt.year

# Pivot to get Dates as rows and Sectors as columns
hhi_pivot = master_hhi.pivot(index='Date', columns='Sector', values='HHI').fillna(0)

supplier_hhi_records = []

# 2. Iterate through every single month in your panel
for current_date, hhi_row in hhi_pivot.iterrows():
    current_year = current_date.year
    
    # Apply a 1-year lag for predictive out-of-sample validity
    # We use the previous year's IO table to weight the current year's risk
    weight_year = current_year - 1 
    
    # Fallback to the earliest available weights if we hit the boundary
    if weight_year not in annual_weights:
        weight_year = min(annual_weights.keys())
        
    omega = annual_weights[weight_year]
    
    # 3. Compute SupplierHHI_j,t = Sum( ω_j,k * HHI_k,t )
    # For a specific Output Sector J, we multiply the J column of the omega matrix 
    # by the current month's HHI values of the Input Sectors (K), and sum them.
    for target_sector in hhi_pivot.columns:
        if target_sector in omega.columns:
            # Extract the weights (ω_k) for Sector J
            weights = omega[target_sector] 
            
            # Align the K input sectors between the omega index and HHI columns
            aligned_hhi = hhi_row.reindex(weights.index).fillna(0)
            
            # The calculation!
            supplier_hhi_val = np.dot(weights.values, aligned_hhi.values)
            
            supplier_hhi_records.append({
                'Date': current_date,
                'Sector': target_sector,
                'Supplier_HHI': supplier_hhi_val
            })

# Convert to final DataFrame and save
supplier_hhi_df = pd.DataFrame(supplier_hhi_records)
supplier_hhi_df.to_csv("./Data/Network_Supplier_HHI.csv", index=False)

print("SUCCESS: Upstream Network Fragility (Supplier HHI) calculated and saved.")
supplier_hhi_df.head()

Calculating dynamic Supplier HHI...
SUCCESS: Upstream Network Fragility (Supplier HHI) calculated and saved.


,Date,Sector,Supplier_HHI
0,2005-01-31,Communication,0.099231
1,2005-01-31,Consumer Discretionary,0.111361
2,2005-01-31,Consumer Staples,0.120242
3,2005-01-31,Energy,0.108432
4,2005-01-31,Financials,0.116705


### OptionMetrics Ivy DB

In [32]:
options_df = pd.read_csv('./Data/Options_Data.csv')
options_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102118 entries, 0 to 102117
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   secid            102118 non-null  int64  
 1   date             102118 non-null  object 
 2   days             102118 non-null  int64  
 3   impl_volatility  102100 non-null  float64
 4   delta            102100 non-null  float64
 5   cp_flag          102118 non-null  object 
 6   cusip            102118 non-null  object 
 7   ticker           102118 non-null  object 
 8   index_flag       102118 non-null  int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 7.0+ MB


In [34]:
options_df.sample(n=50).to_csv("Options_Data_Sample.csv", index=False)

In [36]:
print("Processing Raw OptionMetrics Ivy DB Data...")

# 1. Load the raw downloaded OptionMetrics CSV
options_raw = pd.read_csv('./Data/Options_Data.csv') 

# 2. Format Dates and Tickers
options_raw = options_raw[options_raw['days'] == 30].copy()
options_raw['date'] = pd.to_datetime(options_raw['date'])
options_raw['YearMonth'] = options_raw['date'].dt.to_period('M')
options_raw['ticker'] = options_raw['ticker'] + ' US Equity' 

# 3. Isolate End-of-Month trading days
eom_dates = options_raw.groupby(['YearMonth', 'ticker'])['date'].max().reset_index()
options_eom = pd.merge(options_raw, eom_dates, on=['YearMonth', 'ticker', 'date'])

# 4. Extract specific delta contracts dynamically
options_records = []
grouped = options_eom.groupby(['YearMonth', 'ticker'])

for (ym, ticker), group in grouped:
    try:
        # Separate calls and puts, immediately dropping any NaN rows to avoid the FutureWarning
        puts = group[group['cp_flag'] == 'P'].dropna(subset=['delta', 'impl_volatility']).copy()
        calls = group[group['cp_flag'] == 'C'].dropna(subset=['delta', 'impl_volatility']).copy()
        
        # Safely find IV for 25-Delta OTM options using idxmin()
        if not puts.empty:
            put_25_idx = (puts['delta'] - (-0.25)).abs().idxmin()
            put_25_iv = puts.loc[put_25_idx, 'impl_volatility']
        else:
            put_25_iv = np.nan
            
        if not calls.empty:
            call_25_idx = (calls['delta'] - 0.25).abs().idxmin()
            call_25_iv = calls.loc[call_25_idx, 'impl_volatility']
        else:
            call_25_iv = np.nan
            
        # Safely find IV for 50-Delta ATM options using idxmin()
        if not puts.empty:
            put_50_idx = (puts['delta'] - (-0.50)).abs().idxmin()
            put_50_iv = puts.loc[put_50_idx, 'impl_volatility']
        else:
            put_50_iv = np.nan
            
        if not calls.empty:
            call_50_idx = (calls['delta'] - 0.50).abs().idxmin()
            call_50_iv = calls.loc[call_50_idx, 'impl_volatility']
        else:
            call_50_iv = np.nan
        
        options_records.append({
            'YearMonth': ym,
            'Ticker': ticker,
            'Put_25_IV': put_25_iv,
            'Call_25_IV': call_25_iv,
            'Put_50_IV': put_50_iv,
            'Call_50_IV': call_50_iv,
            'Risk_Reversal_25': put_25_iv - call_25_iv,  # Main Dependent Variable
            'ATM_IV': (put_50_iv + call_50_iv) / 2 # OLD: RR_50 put_50_iv - call_50_iv (ATM baseline proxy)
        })
    except Exception as e:
        continue

# 5. Save the clean processed dataset
options_clean = pd.DataFrame(options_records)

# Drop the raw intermediate IV columns to keep the final merge clean
options_clean = options_clean[['YearMonth', 'Ticker', 'Risk_Reversal_25', 'ATM_IV']]
options_clean.to_csv("./Data/Options_Risk_Reversal_OptionMetrics.csv", index=False)

print(f"OptionMetrics processing complete! Cleaned data saved. Shape: {options_clean.shape}")

Processing Raw OptionMetrics Ivy DB Data...
OptionMetrics processing complete! Cleaned data saved. Shape: (2438, 4)


### Panel Dataset

In [39]:
import pandas as pd
import glob
import os
import numpy as np

print("Initiating Master Panel Merge...")

# 1. Map Sectors to Tickers for cross-referencing
spdr_etfs = {
    'Technology': 'XLK US Equity', 'Health Care': 'XLV US Equity',
    'Energy': 'XLE US Equity', 'Financials': 'XLF US Equity',
    'Industrials': 'XLI US Equity', 'Consumer Staples': 'XLP US Equity',
    'Consumer Discretionary': 'XLY US Equity', 'Utilities': 'XLU US Equity',
    'Materials': 'XLB US Equity', 'Real Estate': 'XLRE US Equity',
    'Communication': 'XLC US Equity'
}

# --- A. Process the HHI Files (Base Panel) ---
hhi_files = glob.glob('./Data/HHI/*_HHI.csv')
hhi_list = []
for file in hhi_files:
    df = pd.read_csv(file)
    hhi_list.append(df)

panel_df = pd.concat(hhi_list, ignore_index=True)
panel_df['YearMonth'] = pd.to_datetime(panel_df['Date']).dt.to_period('M')
panel_df['Ticker'] = panel_df['Sector'].map(spdr_etfs)
panel_df = panel_df.drop(columns=['Date', 'Index_Ticker']) # Keep YearMonth as the key


# --- B. Add Network Supplier HHI ---
supplier_df = pd.read_csv('./Data/Network_Supplier_HHI.csv')
supplier_df['YearMonth'] = pd.to_datetime(supplier_df['Date']).dt.to_period('M')
supplier_df = supplier_df.drop(columns=['Date'])

panel_df = pd.merge(panel_df, supplier_df, on=['YearMonth', 'Sector'], how='left')


# --- C. Add OptionMetrics Data (Dependent Variable) ---
options_df = pd.read_csv('./Data/Options_Risk_Reversal_OptionMetrics.csv')
# Since it was saved as a string, convert it back to Period type for merging
options_df['YearMonth'] = pd.PeriodIndex(options_df['YearMonth'], freq='M')

panel_df = pd.merge(panel_df, options_df, on=['YearMonth', 'Ticker'], how='left')


# --- D. Add Sector Controls (Requires Unpivoting) ---
controls_df = pd.read_csv('./Data/Sector_Controls.csv')
controls_df['YearMonth'] = pd.to_datetime(controls_df['date']).dt.to_period('M')
controls_df = controls_df.drop(columns=['date'])

# Melt from Wide to Long
controls_melt = controls_df.melt(id_vars=['YearMonth'], var_name='Metric', value_name='Value')

# Split 'XLK US Equity_pe_ratio' into 'XLK US Equity' and 'pe_ratio'
controls_melt[['Ticker', 'Variable']] = controls_melt['Metric'].str.split('_', n=1, expand=True)

# Pivot back so PE_Ratio and Returns are their own columns
controls_long = controls_melt.pivot(index=['YearMonth', 'Ticker'], columns='Variable', values='Value').reset_index()

panel_df = pd.merge(panel_df, controls_long, on=['YearMonth', 'Ticker'], how='left')


# --- E. Add Macro Controls (FRED) ---
macro_df = pd.read_csv('./Data/Macro_Controls_FRED.csv')
macro_df['YearMonth'] = pd.to_datetime(macro_df['DATE']).dt.to_period('M')
macro_df = macro_df.drop(columns=['DATE'])

panel_df = pd.merge(panel_df, macro_df, on=['YearMonth'], how='left')


# --- F. Add Hub Shock ---
''' OLD CASCADE PROXY
cascade_df = pd.read_csv('./Data/Sector_Cascade_Proxy.csv')
cascade_df['YearMonth'] = pd.to_datetime(cascade_df['Date']).dt.to_period('M')
cascade_df = cascade_df.drop(columns=['Date'])

# Merge precisely on BOTH YearMonth and Sector to map local shocks to the correct ETF
panel_df = pd.merge(panel_df, cascade_df, on=['YearMonth', 'Sector'], how='left')
'''
hubshock_df = pd.read_csv('./Data/Sector_Hub_Shock.csv')
hubshock_df['YearMonth'] = pd.to_datetime(hubshock_df['Date']).dt.to_period('M')
hubshock_df = hubshock_df.drop(columns=['Date'])

# Merge precisely on BOTH YearMonth and Sector to map local shocks to the correct ETF
panel_df = pd.merge(panel_df, hubshock_df, on=['YearMonth', 'Sector'], how='left')


# --- G. Final Cleanup ---
# Reorder columns for a clean econometric view
front_cols = [
    'YearMonth', 'Sector', 'Ticker', 
    'Risk_Reversal_25', 'ATM_IV', 
    'HHI', 'Supplier_HHI', 'Hub_Shock'
]
other_cols = [c for c in panel_df.columns if c not in front_cols]
panel_df = panel_df[front_cols + other_cols]

# Sort chronologically and by sector
panel_df = panel_df.sort_values(by=['Sector', 'YearMonth']).reset_index(drop=True)

# Export Master Panel
panel_df.to_csv("Master_Predictive_Panel.csv", index=False)

print(f"Merge Complete! Final dataset shape: {panel_df.shape}")
panel_df.info()

Initiating Master Panel Merge...
Merge Complete! Final dataset shape: (2794, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2794 entries, 0 to 2793
Data columns (total 12 columns):
 #   Column                       Non-Null Count  Dtype    
---  ------                       --------------  -----    
 0   YearMonth                    2794 non-null   period[M]
 1   Sector                       2794 non-null   object   
 2   Ticker                       2794 non-null   object   
 3   Risk_Reversal_25             2437 non-null   float64  
 4   ATM_IV                       2437 non-null   float64  
 5   HHI                          2794 non-null   float64  
 6   Supplier_HHI                 2794 non-null   float64  
 7   Hub_Shock                    2794 non-null   int64    
 8   tot_return_index_gross_dvds  2504 non-null   float64  
 9   VIX                          2794 non-null   float64  
 10  Term_Spread                  2794 non-null   float64  
 11  Credit_Spread              

In [41]:
panel_df.sample(n=50).to_csv("Panel_Sample.csv", index=False)

## Models

In [44]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, compare
from scipy import stats
from sklearn.metrics import mean_squared_error

In [46]:
# 1. Load Master Panel
df = pd.read_csv('Master_Predictive_Panel.csv')
df['YearMonth'] = pd.to_datetime(df['YearMonth'])

# 2. Build forward-shifted dependent variable (predict t+1 from period t)
df.sort_values(by=['Ticker', 'YearMonth'], inplace=True)
df['Fwd_RR25'] = df.groupby('Ticker')['Risk_Reversal_25'].shift(-1)

# 3. Set MultiIndex required by PanelOLS
df.set_index(['Ticker', 'YearMonth'], inplace=True)

# 4. Drop rows missing any required regressor or dependent variable
required = ['Fwd_RR25', 'ATM_IV', 'tot_return_index_gross_dvds', 'Hub_Shock']
df_clean = df.dropna(subset=required).copy()

# 5. Build sector dummies and interaction columns once for reuse across all models
sector_dummies = pd.get_dummies(df_clean['Sector'], drop_first=True, dtype=float)

hhi_interact_cols = []
supp_interact_cols = []
for col in sector_dummies.columns:
    hhi_name = f'HHI_x_{col}'
    supp_name = f'SuppHHI_x_{col}'
    df_clean[hhi_name] = df_clean['HHI'] * sector_dummies[col]
    df_clean[supp_name] = df_clean['Supplier_HHI'] * sector_dummies[col]
    hhi_interact_cols.append(hhi_name)
    supp_interact_cols.append(supp_name)

controls = ['ATM_IV', 'tot_return_index_gross_dvds']

print(f'Panel shape after cleaning: {df_clean.shape}')
print(f'Sectors: {df_clean["Sector"].nunique()}, Time periods: {df_clean.index.get_level_values("YearMonth").nunique()}')

Panel shape after cleaning: (2425, 31)
Sectors: 11, Time periods: 247


### In-Sample Inference

Five specifications, all with two-way fixed effects and entity-clustered standard errors:

- **M1 (Baseline Structural):** $RR_{j,t+1} = \alpha_j + \gamma_t + \beta_1 HHI_{j,t} + \beta_2 SupplierHHI_{j,t} + \theta'X_{j,t} + \varepsilon_{j,t+1}$
- **M2 (Additive Hub Shock):** $RR_{j,t+1} = \alpha_j + \gamma_t + \beta_1 HHI_{j,t} + \beta_2 HubShock_{j,t} + \theta'X_{j,t} + \varepsilon_{j,t+1}$. The original interaction term $HHI \times Cascade$ was dropped on $t = -0.41$, so this is the additive level-shift specification. The trigger is now constructed via the rolling 36-month heavyweight window described in the data collection script and renamed Hub Shock per advisor feedback.
- **M3A (Heterogeneous HHI):** sector-interacted internal HHI with homogeneous Supplier HHI.
- **M3B (Kitchen Sink):** both internal HHI and Supplier HHI fully interacted with sector dummies.
- **M3C (Alternating):** homogeneous internal HHI with sector-interacted Supplier HHI.

In [49]:
# Model 1: Baseline Structural
exog1 = sm.add_constant(df_clean[['HHI', 'Supplier_HHI'] + controls])
res1 = PanelOLS(df_clean['Fwd_RR25'], exog1, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

# Model 2: Additive Hub Shock (revised, no interaction with HHI)
exog2 = sm.add_constant(df_clean[['HHI', 'Hub_Shock'] + controls])
res2 = PanelOLS(df_clean['Fwd_RR25'], exog2, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

# Model 3A: Heterogeneous HHI + Homogeneous Supplier_HHI
exog3a = sm.add_constant(
    df_clean[['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + controls]
)
res3a = PanelOLS(df_clean['Fwd_RR25'], exog3a, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

# Model 3B: Kitchen Sink (heterogeneous HHI AND heterogeneous Supplier_HHI)
exog3b = sm.add_constant(
    df_clean[['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + supp_interact_cols + controls]
)
res3b = PanelOLS(df_clean['Fwd_RR25'], exog3b, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

# Model 3C: Alternating (homogeneous HHI + heterogeneous Supplier_HHI)
exog3c = sm.add_constant(
    df_clean[['HHI', 'Supplier_HHI'] + supp_interact_cols + controls]
)
res3c = PanelOLS(df_clean['Fwd_RR25'], exog3c, entity_effects=True, time_effects=True).fit(
    cov_type='clustered', cluster_entity=True
)

results_dict = {
    'M1: Baseline':       res1,
    'M2: Hub Shock':      res2,
    'M3A: Het. HHI':      res3a,
    'M3B: Kitchen Sink':  res3b,
    'M3C: Alternating':   res3c,
}
comp = compare(results_dict, stars=True)
print(comp.summary)

# Export comparison table for the thesis document
with open('./Output/regression_results.tex', 'w') as f:
    f.write(comp.summary.as_latex())
with open('./Output/regression_results.html', 'w') as f:
    f.write(comp.summary.as_html())

                                                    Model Comparison                                                   
                                         M1: Baseline  M2: Hub Shock   M3A: Het. HHI M3B: Kitchen Sink M3C: Alternating
-----------------------------------------------------------------------------------------------------------------------
Dep. Variable                                Fwd_RR25       Fwd_RR25        Fwd_RR25          Fwd_RR25         Fwd_RR25
Estimator                                    PanelOLS       PanelOLS        PanelOLS          PanelOLS         PanelOLS
No. Observations                                 2425           2425            2425              2425             2425
Cov. Est.                                   Clustered      Clustered       Clustered         Clustered        Clustered
R-squared                                      0.0075         0.0028          0.0117            0.0194           0.0142
R-Squared (Within)                      

### Out-of-Sample Forecasting

Three challengers compared against a baseline with macro and sector controls only.

- **Baseline:** term spread, credit spread, ATM IV, total return index. Entity FE only.
- **Challenger H (Homogeneous):** baseline + HHI + Supplier HHI. Original specification.
- **Challenger 3A (Heterogeneous HHI):** baseline + HHI + sector-interacted HHI + Supplier HHI. Matches Model 3A.
- **Challenger 3B (Kitchen Sink):** baseline + heterogeneous HHI + heterogeneous Supplier HHI. Matches Model 3B.

Train: Jan 2005 to Dec 2019. Test: Jan 2025 to Feb 2026. The 2020-2024 period is excluded from both.

**Methodology note (Walker reframe):** under the non-monotonic fragility framing, contemporaneous HHI captures the current structural state of the network, which is what determines current fragility. The forward-shifted dependent variable already ensures the relationship is predictive rather than contemporaneous, so a distributed-lag specification on HHI is not needed on the critical path. A single-period lag remains an optional appendix robustness exercise but is not run here.

In [64]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS
from scipy import stats

# ----- Fixed train/test split (matches methodology section) -----
df_oos = df_clean.reset_index()
train = df_oos[(df_oos['YearMonth'] >= '2005-01-01') &
               (df_oos['YearMonth'] <= '2019-12-31')] \
        .set_index(['Ticker', 'YearMonth'])
test  = df_oos[(df_oos['YearMonth'] >= '2025-01-01') &
               (df_oos['YearMonth'] <= '2026-02-28')] \
        .set_index(['Ticker', 'YearMonth'])

print(f'Train: {train.shape[0]} obs, {train.index.get_level_values("Ticker").nunique()} entities, '
      f'{train.index.get_level_values("YearMonth").nunique()} months')
print(f'Test:  {test.shape[0]} obs, {test.index.get_level_values("Ticker").nunique()} entities, '
      f'{test.index.get_level_values("YearMonth").nunique()} months')

forecast_controls = ['ATM_IV', 'tot_return_index_gross_dvds',
                     'Term_Spread', 'Credit_Spread']

def filter_identified(train_df, regressors, tol=1e-10):
    """
    Keep regressors that have meaningful within-entity variation in the
    training set. A column is dropped if it is zero everywhere or if its
    within-entity standard deviation is zero for every entity (which makes
    it collinear with entity fixed effects).
    """
    active = []
    for c in regressors:
        vals = train_df[c]
        if np.allclose(vals.values, 0):
            continue
        within_std = vals.groupby(level='Ticker').std(ddof=0)
        if (within_std > tol).any():
            active.append(c)
    return active

def fit_and_predict_fixed(train_df, test_df, regressors, dep='Fwd_RR25'):
    train_df = train_df.dropna(subset=regressors + [dep]).copy()
    test_df  = test_df.dropna(subset=regressors + [dep]).copy()

    active = filter_identified(train_df, regressors)
    X_train = sm.add_constant(train_df[active], has_constant='add')
    X_test  = sm.add_constant(test_df[active],  has_constant='add')

    res = PanelOLS(train_df[dep], X_train,
                   entity_effects=True, time_effects=False,
                   check_rank=False).fit()

    params = res.params
    pred_xb = X_test[params.index].values @ params.values

    eff = res.estimated_effects
    if isinstance(eff, pd.DataFrame):
        eff = eff.iloc[:, 0]
    entity_fe = eff.groupby(level='Ticker').first()
    fe_vals = entity_fe.reindex(
        test_df.index.get_level_values('Ticker')).values
    fe_vals = np.nan_to_num(fe_vals, nan=0.0)

    pred = pd.Series(pred_xb + fe_vals, index=test_df.index)
    err  = test_df[dep] - pred
    return res, pred, err

oos_specs = {
    'Baseline':                forecast_controls,
    'Homogeneous (H)':        ['HHI', 'Supplier_HHI'] + forecast_controls,
    'Heterogeneous HHI (3A)': ['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + forecast_controls,
    'Kitchen Sink (3B)':      ['HHI'] + hhi_interact_cols + ['Supplier_HHI']
                              + supp_interact_cols + forecast_controls,
}

errors = {}
for name, regs in oos_specs.items():
    _, _, err = fit_and_predict_fixed(train, test, regs)
    errors[name] = err
    print(f'{name:30s} N forecasts: {len(err)}')

# ----- Diebold-Mariano on common forecast sample -----
def diebold_mariano(e_base, e_chal, h=1):
    common = e_base.index.intersection(e_chal.index)
    d = (e_base.loc[common] ** 2) - (e_chal.loc[common] ** 2)
    d_t = d.groupby(level='YearMonth').mean()
    T = len(d_t)
    d_bar = d_t.mean()
    gamma_0 = ((d_t - d_bar) ** 2).sum() / T
    var_d = gamma_0
    for k in range(1, h):
        gamma_k = ((d_t.iloc[k:] - d_bar) *
                   (d_t.iloc[:-k].values - d_bar)).sum() / T
        var_d += 2 * (1 - k / h) * gamma_k
    dm = d_bar / np.sqrt(var_d / T)
    p  = 2 * (1 - stats.norm.cdf(abs(dm)))
    return dm, p, T

base = errors['Baseline']
rows = []
for name, e in errors.items():
    common = base.index.intersection(e.index)
    mspe = (e.loc[common] ** 2).mean()
    if name == 'Baseline':
        rows.append({'Model': name, 'N': len(common), 'MSPE': mspe,
                     'Pct vs Baseline': np.nan, 'DM stat': np.nan, 'DM p': np.nan})
    else:
        mspe_b = (base.loc[common] ** 2).mean()
        pct = (mspe - mspe_b) / mspe_b * 100
        dm, p, _ = diebold_mariano(base, e, h=1)
        rows.append({'Model': name, 'N': len(common), 'MSPE': mspe,
                     'Pct vs Baseline': pct, 'DM stat': dm, 'DM p': p})

oos_table = pd.DataFrame(rows)
print('\n', oos_table.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
oos_table.to_csv('./Output/oos_fixed_window.csv', index=False)
oos_table.to_latex('./Output/oos_fixed_window.tex', index=False, float_format='%.4f')

Train: 1688 obs, 11 entities, 180 months
Test:  77 obs, 11 entities, 7 months
Baseline                       N forecasts: 77
Homogeneous (H)                N forecasts: 77
Heterogeneous HHI (3A)         N forecasts: 77
Kitchen Sink (3B)              N forecasts: 77

                  Model  N     MSPE  Pct vs Baseline   DM stat     DM p
              Baseline 77 0.000078              NaN       NaN      NaN
       Homogeneous (H) 77 0.000082         5.238646 -2.521340 0.011691
Heterogeneous HHI (3A) 77 0.000099        26.941264 -1.409273 0.158755
     Kitchen Sink (3B) 77 0.000112        43.761078 -1.579281 0.114272


In [70]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# ----- Restrict to the methodology training window -----
df_cv = df_clean.reset_index()
train_full = df_cv[(df_cv['YearMonth'] >= '2005-01-01') &
                   (df_cv['YearMonth'] <= '2019-12-31')].copy()
train_full['Year'] = train_full['YearMonth'].dt.year
years = sorted(train_full['Year'].unique())
print(f'Training window: {len(years)} years, {train_full.shape[0]} obs')

# ----- Specification dictionary (matches in-sample inference) -----
controls = ['ATM_IV', 'tot_return_index_gross_dvds']
specs = {
    'M1':  ['HHI', 'Supplier_HHI'] + controls,
    'M3A': ['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + controls,
    'M3B': ['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + supp_interact_cols + controls,
    'M3C': ['HHI', 'Supplier_HHI'] + supp_interact_cols + controls,
}

# Coefficients we want to track. These are the ones the paper's
# argument actually rests on. The full coefficient vector is also
# saved per fold so you can inspect anything else later.
key_vars = {
    'M1':  ['Supplier_HHI'],
    'M3A': ['HHI', 'Supplier_HHI',
            'HHI_x_Real Estate', 'HHI_x_Health Care',
            'HHI_x_Industrials', 'HHI_x_Energy'],
    'M3B': ['SuppHHI_x_Real Estate', 'SuppHHI_x_Financials',
            'SuppHHI_x_Health Care', 'SuppHHI_x_Industrials'],
    'M3C': ['SuppHHI_x_Real Estate', 'SuppHHI_x_Industrials'],
}

def fit_two_way(data, regressors, dep='Fwd_RR25'):
    data = data.dropna(subset=regressors + [dep]).copy()
    data = data.set_index(['Ticker', 'YearMonth'])
    X = sm.add_constant(data[regressors], has_constant='add')
    return PanelOLS(data[dep], X,
                    entity_effects=True, time_effects=True,
                    check_rank=False).fit(cov_type='clustered',
                                          cluster_entity=True)

# ----- Reference fits on full training window -----
ref_fits = {name: fit_two_way(train_full, regs) for name, regs in specs.items()}

# ----- Leave-one-year-out loop -----
fold_records = {name: [] for name in specs}
for y in years:
    fold = train_full[train_full['Year'] != y]
    for name, regs in specs.items():
        try:
            res = fit_two_way(fold, regs)
            row = {'held_out': y}
            for v in res.params.index:
                row[f'{v}__coef'] = res.params[v]
                row[f'{v}__t']    = res.tstats[v]
            fold_records[name].append(row)
        except Exception as exc:
            print(f'  {name}, year {y}: {type(exc).__name__}: {str(exc)[:120]}')

# ----- Stability summary -----
stability_rows = []
for name, recs in fold_records.items():
    fold_df = pd.DataFrame(recs)
    ref = ref_fits[name].params
    ref_t = ref_fits[name].tstats

    for v in key_vars[name]:
        if v not in ref.index:
            print(f'  {name}: tracked variable {v!r} not in reference params, skipping.')
            continue
        col = f'{v}__coef'
        tcol = f'{v}__t'
        if col not in fold_df.columns:
            continue

        coefs = fold_df[col].dropna()
        ts    = fold_df[tcol].dropna()
        full_sign = np.sign(ref[v])
        same_sign = (np.sign(coefs) == full_sign).mean()
        sig_5pct  = (ts.abs() >= 1.96).mean()

        stability_rows.append({
            'Model':       name,
            'Variable':    v,
            'Full coef':   ref[v],
            'Full t':      ref_t[v],
            'Median coef': coefs.median(),
            'Min coef':    coefs.min(),
            'Max coef':    coefs.max(),
            'IQR width':   coefs.quantile(0.75) - coefs.quantile(0.25),
            'Same sign %': 100 * same_sign,
            'Sig at 5%':   100 * sig_5pct,
            'Folds':       len(coefs),
        })

stability = pd.DataFrame(stability_rows)
print('\n=== Leave-one-year-out coefficient stability ===\n')
print(stability.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
stability.to_csv('./Output/cv_loo_stability.csv', index=False)

# ----- Optional: full per-fold panel for any deeper inspection -----
for name, recs in fold_records.items():
    pd.DataFrame(recs).to_csv(f'./Output/cv_loo_folds/cv_loo_{name}_folds.csv', index=False)

Training window: 15 years, 1688 obs

=== Leave-one-year-out coefficient stability ===

Model              Variable  Full coef  Full t  Median coef  Min coef  Max coef  IQR width  Same sign %  Sig at 5%  Folds
   M1          Supplier_HHI    -0.0694 -4.9591      -0.0706   -0.0918   -0.0475    +0.0034    +100.0000  +100.0000     15
  M3A                   HHI    -0.2860 -5.4698      -0.2838   -0.3685   +0.1239    +0.0314     +93.3333   +80.0000     15
  M3A          Supplier_HHI    -0.0656 -3.3838      -0.0663   -0.0798   -0.0377    +0.0060    +100.0000   +93.3333     15
  M3A     HHI_x_Real Estate    +0.7055 +1.8122      +0.6863  -13.5291   +4.7740    +0.1551     +93.3333   +26.6667     15
  M3A     HHI_x_Health Care    +0.3722 +4.2387      +0.3727   -0.0324   +0.4856    +0.0932     +93.3333   +80.0000     15
  M3A     HHI_x_Industrials    +0.3164 +5.6118      +0.3202   -0.0919   +0.4053    +0.0538     +93.3333   +80.0000     15
  M3A          HHI_x_Energy    +0.3052 +5.2313      +0.3018

In [72]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

# linearmodels exposes Driscoll-Kraay as cov_type='kernel' with kernel='bartlett'
# applied to a pooled regression with appropriate FE. The bandwidth controls
# how many lags of the cross-sectionally averaged scores are used in the HAC
# variance estimator.

dk_specs = {
    'M1':  ['HHI', 'Supplier_HHI'] + controls,
    'M2':  ['HHI', 'Hub_Shock'] + controls,
    'M3A': ['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + controls,
    'M3B': ['HHI'] + hhi_interact_cols + ['Supplier_HHI']
                   + supp_interact_cols + controls,
    'M3C': ['HHI', 'Supplier_HHI'] + supp_interact_cols + controls,
}

def fit_dk(regressors, bandwidth, dep='Fwd_RR25', data=df_clean):
    data = data.dropna(subset=regressors + [dep]).copy()
    X = sm.add_constant(data[regressors], has_constant='add')
    return PanelOLS(data[dep], X,
                    entity_effects=True, time_effects=True,
                    check_rank=False).fit(
        cov_type='kernel', kernel='bartlett',
        bandwidth=bandwidth)

# ----- Estimate at two bandwidths to flag bandwidth-sensitive coefs -----
bandwidths = {'DK6': 6, 'DK12': 12}

# Variables we report on. These are the same set tracked in LOO plus the
# additive Hub_Shock from M2 and the homogeneous HHI/Supplier_HHI from M3B
# and M3C, so the reader can see how every coefficient referenced in the
# Results section behaves under DK SEs.
report_vars = {
    'M1':  ['HHI', 'Supplier_HHI'],
    'M2':  ['HHI', 'Hub_Shock'],
    'M3A': ['HHI', 'Supplier_HHI',
            'HHI_x_Consumer Discretionary', 'HHI_x_Consumer Staples',
            'HHI_x_Energy', 'HHI_x_Financials', 'HHI_x_Health Care',
            'HHI_x_Industrials', 'HHI_x_Materials', 'HHI_x_Real Estate',
            'HHI_x_Technology', 'HHI_x_Utilities'],
    'M3B': ['HHI', 'Supplier_HHI',
            'SuppHHI_x_Real Estate', 'SuppHHI_x_Financials',
            'SuppHHI_x_Health Care', 'SuppHHI_x_Industrials'],
    'M3C': ['HHI', 'Supplier_HHI',
            'SuppHHI_x_Real Estate', 'SuppHHI_x_Industrials'],
}

# Reference clustered-SE results for side-by-side comparison
def fit_clustered(regressors, dep='Fwd_RR25', data=df_clean):
    data = data.dropna(subset=regressors + [dep]).copy()
    X = sm.add_constant(data[regressors], has_constant='add')
    return PanelOLS(data[dep], X,
                    entity_effects=True, time_effects=True,
                    check_rank=False).fit(cov_type='clustered',
                                          cluster_entity=True)

rows = []
for name, regs in dk_specs.items():
    res_cl  = fit_clustered(regs)
    res_dk6 = fit_dk(regs, bandwidth=6)
    res_dk12 = fit_dk(regs, bandwidth=12)

    for v in report_vars[name]:
        if v not in res_cl.params.index:
            continue
        rows.append({
            'Model':        name,
            'Variable':     v,
            'Coef':         res_cl.params[v],
            'Clustered t':  res_cl.tstats[v],
            'DK6 t':        res_dk6.tstats[v],
            'DK12 t':       res_dk12.tstats[v],
            'Sig (clust)':  '***' if abs(res_cl.tstats[v])  >= 2.576
                            else ('**' if abs(res_cl.tstats[v])  >= 1.96
                                  else ('*' if abs(res_cl.tstats[v])  >= 1.645 else '')),
            'Sig (DK6)':    '***' if abs(res_dk6.tstats[v])  >= 2.576
                            else ('**' if abs(res_dk6.tstats[v])  >= 1.96
                                  else ('*' if abs(res_dk6.tstats[v])  >= 1.645 else '')),
            'Sig (DK12)':   '***' if abs(res_dk12.tstats[v]) >= 2.576
                            else ('**' if abs(res_dk12.tstats[v]) >= 1.96
                                  else ('*' if abs(res_dk12.tstats[v]) >= 1.645 else '')),
        })

dk_table = pd.DataFrame(rows)
print('\n=== Driscoll-Kraay vs Entity-Clustered Standard Errors ===\n')
print(dk_table.to_string(index=False, float_format=lambda x: f'{x:+.4f}'))
dk_table.to_csv('./Output/dk_robustness.csv', index=False)
dk_table.to_latex('./Output/dk_robustness.tex', index=False, float_format='%.3f')

# ----- Quick flag for any sign reversal of significance at 5% -----
flips = dk_table[
    (dk_table['Sig (clust)'].str.contains(r'\*\*', regex=True))
    & (~dk_table['Sig (DK6)'].str.contains(r'\*\*', regex=True))
]
if not flips.empty:
    print('\nCoefficients significant under clustered SEs but NOT under DK6:')
    print(flips[['Model', 'Variable', 'Clustered t', 'DK6 t']].to_string(index=False))
else:
    print('\nNo significance reversals at 5% between clustered and DK6 SEs.')


=== Driscoll-Kraay vs Entity-Clustered Standard Errors ===

Model                     Variable    Coef  Clustered t   DK6 t  DK12 t Sig (clust) Sig (DK6) Sig (DK12)
   M1                          HHI -0.0073      -1.0638 -0.7028 -0.7369                                 
   M1                 Supplier_HHI -0.0758      -5.2227 -2.4340 -2.4439         ***        **         **
   M2                          HHI +0.0023      +0.2084 +0.2171 +0.2103                                 
   M2                    Hub_Shock +0.0014      +2.5264 +1.8957 +1.8185          **         *          *
  M3A                          HHI -0.0859      -4.9424 -2.5883 -2.6730         ***       ***        ***
  M3A                 Supplier_HHI -0.0632      -4.6469 -1.7765 -1.7973         ***         *          *
  M3A HHI_x_Consumer Discretionary +0.0592      +3.2007 +1.6036 +1.6462         ***                    *
  M3A       HHI_x_Consumer Staples +0.1007      +3.8981 +1.9045 +1.8116         ***         *      

##

## Figures

In [76]:
"""
Coefficient magnitude analysis: translate the M3A HHI loadings into
economically interpretable basis-point movements in the forward 25-delta
risk reversal per one-standard-deviation increase in sector HHI.

INPUTS:
    Master_Predictive_Panel.csv  (set PANEL_PATH below)

OUTPUTS:
    table5_coefficient_magnitudes.tex   (LaTeX-ready)
    coefficient_magnitudes.csv          (raw numbers for cross-checking)

Computes, for each sector:
    delta_RR_bps_per_1SD = (baseline + interaction) * SD(HHI_j) * 10000

where 10000 converts the unit (decimal RR) to basis points. Note that the
RR is already in decimal form (e.g. 0.05 = 500 bps of IV difference between
25-delta put and call), so a coefficient times an HHI change gives a
predicted change in the same decimal units.

USAGE:
    python analysis_coefficient_magnitudes.py
"""

import os
import sys
import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# Configuration: edit PANEL_PATH if needed
# -----------------------------------------------------------------------------
PANEL_PATH = "Master_Predictive_Panel.csv"
OUT_TEX    = "./Tables/table5_coefficient_magnitudes.tex"
# OUT_CSV    = "coefficient_magnitudes.csv"

# M3A coefficients (from the in-sample experiment file).
# Reference (omitted) sector is Communication; its net effect equals BASELINE.
BASELINE = -0.0859

INTERACTIONS = {
    "Real Estate":            (0.2976, "***"),
    "Health Care":            (0.1627, "***"),
    "Utilities":              (0.1377, ""   ),
    "Materials":              (0.1246, "***"),
    "Industrials":            (0.1096, "***"),
    "Energy":                 (0.1074, "***"),
    "Consumer Staples":       (0.1007, "***"),
    "Consumer Discretionary": (0.0592, "***"),
    "Financials":             (0.0388, ""   ),
    "Technology":             (0.0131, ""   ),
    "Communication":          (0.0000, "ref"),  # reference: net effect = baseline
}

# -----------------------------------------------------------------------------
# Load panel and compute per-sector HHI standard deviations
# -----------------------------------------------------------------------------
if not os.path.exists(PANEL_PATH):
    print(f"ERROR: panel file not found at {PANEL_PATH!r}.")
    print("Edit PANEL_PATH at the top of this script to point at your master panel.")
    sys.exit(1)

panel = pd.read_csv(PANEL_PATH)
required = {"Sector", "HHI"}
missing = required - set(panel.columns)
if missing:
    print(f"ERROR: panel is missing required columns {missing}.")
    sys.exit(1)

# Within-sector standard deviation of HHI over the panel
sd_hhi = panel.groupby("Sector")["HHI"].std(ddof=1)
mean_hhi = panel.groupby("Sector")["HHI"].mean()
n_obs = panel.groupby("Sector")["HHI"].count()

# -----------------------------------------------------------------------------
# Build the magnitude table
# -----------------------------------------------------------------------------
rows = []
sectors_order = [
    "Real Estate", "Health Care", "Utilities", "Materials", "Industrials",
    "Energy", "Consumer Staples", "Consumer Discretionary", "Financials",
    "Technology", "Communication",
]

for sector in sectors_order:
    if sector not in sd_hhi.index:
        print(f"  Skipping {sector}: not present in panel")
        continue

    interact_coef, sig = INTERACTIONS[sector]
    net_coef = BASELINE + interact_coef

    sd = sd_hhi[sector]
    mean = mean_hhi[sector]
    n = n_obs[sector]

    # Predicted change in RR (decimal) for a 1-SD increase in HHI
    delta_rr = net_coef * sd
    # Convert decimal IV difference to basis points (1.0 RR = 10,000 bps)
    delta_rr_bps = delta_rr * 10000

    rows.append({
        "Sector":             sector,
        "N":                  int(n),
        "Mean(HHI)":          mean,
        "SD(HHI)":            sd,
        "Net coef":           net_coef,
        "Delta RR (decimal)": delta_rr,
        "Delta RR (bps)":     delta_rr_bps,
        "Interaction sig":    sig,
    })

mag = pd.DataFrame(rows)
# mag.to_csv(OUT_CSV, index=False, float_format="%.6f")
# print(f"\nSaved raw magnitudes -> {OUT_CSV}\n")
print(mag.to_string(index=False, float_format=lambda x: f"{x:+.4f}"))

# -----------------------------------------------------------------------------
# Build LaTeX table
# -----------------------------------------------------------------------------
def fmt(x, p=4):
    if pd.isna(x):
        return "---"
    return f"{x:+.{p}f}" if isinstance(x, (float, np.floating)) and x != 0 else f"{x:.{p}f}"

def sig_marker(s):
    return s if s in ("***", "**", "*") else ("ref." if s == "ref" else "")

# Sort by absolute basis-point impact, descending
mag_sorted = mag.reindex(mag["Delta RR (bps)"].abs().sort_values(ascending=False).index)

tex_lines = []
tex_lines.append(r"% Auto-generated by analysis_coefficient_magnitudes.py")
tex_lines.append(r"\begin{table}[!htbp]")
tex_lines.append(r"\centering")
tex_lines.append(r"\small")
tex_lines.append(r"\caption{Economic Magnitude of Net Sector-Specific HHI Effects (Model 3A)}")
tex_lines.append(r"\label{tab:coefficient_magnitudes}")
tex_lines.append(r"\begin{threeparttable}")
tex_lines.append(r"\begin{tabular}{l c c c c c}")
tex_lines.append(r"\toprule")
tex_lines.append(r" & \multicolumn{1}{c}{Mean} & \multicolumn{1}{c}{SD}")
tex_lines.append(r"   & \multicolumn{1}{c}{Net coef.} ")
tex_lines.append(r"   & \multicolumn{1}{c}{$\Delta RR$ per $+1\sigma$ HHI}")
tex_lines.append(r"   & \multicolumn{1}{c}{$\Delta RR$ per $+1\sigma$ HHI} \\")
tex_lines.append(r"Sector & \multicolumn{1}{c}{HHI} & \multicolumn{1}{c}{HHI}")
tex_lines.append(r"   & \multicolumn{1}{c}{(M3A)}")
tex_lines.append(r"   & \multicolumn{1}{c}{(decimal)}")
tex_lines.append(r"   & \multicolumn{1}{c}{(basis points)} \\")
tex_lines.append(r"\midrule")

for _, r in mag_sorted.iterrows():
    sector = r["Sector"]
    sig = sig_marker(r["Interaction sig"])
    sig_tex = (r"$^{" + sig + r"}$") if sig and sig != "ref." else (r" (ref.)" if sig == "ref." else "")
    line = (
        f"{sector}{sig_tex} & "
        f"{r['Mean(HHI)']:.3f} & "
        f"{r['SD(HHI)']:.3f} & "
        f"{r['Net coef']:+.3f} & "
        f"{r['Delta RR (decimal)']:+.4f} & "
        f"{r['Delta RR (bps)']:+.1f} \\\\"
    )
    tex_lines.append(line)

tex_lines.append(r"\bottomrule")
tex_lines.append(r"\end{tabular}")

tex_lines.append(r"\begin{tablenotes}[flushleft]")
tex_lines.append(r"\footnotesize")
tex_lines.append(
    r"\item \textit{Notes.} Net coefficient equals the M3A baseline "
    r"$\hat{\beta}_{\text{HHI}}=-0.086$ plus the sector-specific interaction "
    r"term. Mean and SD of HHI are computed within sector across all "
    r"sample months. $\Delta RR$ per $+1\sigma$ HHI gives the predicted "
    r"change in the one-month-ahead 25-delta risk reversal for a "
    r"one-standard-deviation increase in sector HHI, holding all controls "
    r"and fixed effects constant. Decimal units convert to basis points "
    r"by multiplication by $10{,}000$ (a 25-delta risk reversal of $0.01$ "
    r"corresponds to a $100$ bp put-call IV gap). Significance markers "
    r"refer to the sector \emph{interaction} term: "
    r"$^{*}\,p<0.10$, $^{**}\,p<0.05$, $^{***}\,p<0.01$ "
    r"under entity-clustered standard errors. Sectors are sorted by "
    r"absolute basis-point impact."
)
tex_lines.append(r"\end{tablenotes}")
tex_lines.append(r"\end{threeparttable}")
tex_lines.append(r"\end{table}")

with open(OUT_TEX, "w") as f:
    f.write("\n".join(tex_lines) + "\n")
print(f"\nSaved LaTeX table -> {OUT_TEX}")

                Sector   N  Mean(HHI)  SD(HHI)  Net coef  Delta RR (decimal)  Delta RR (bps) Interaction sig
           Real Estate 254    +0.0548  +0.0047   +0.2117             +0.0010         +9.9011             ***
           Health Care 254    +0.0531  +0.0111   +0.0768             +0.0009         +8.5425             ***
             Utilities 254    +0.0590  +0.0035   +0.0518             +0.0002         +1.8381                
             Materials 254    +0.0765  +0.0164   +0.0387             +0.0006         +6.3439             ***
           Industrials 254    +0.0524  +0.0316   +0.0237             +0.0007         +7.4893             ***
                Energy 254    +0.1760  +0.0453   +0.0215             +0.0010         +9.7501             ***
      Consumer Staples 254    +0.0930  +0.0222   +0.0148             +0.0003         +3.2903             ***
Consumer Discretionary 254    +0.1022  +0.0501   -0.0267             -0.0013        -13.3800             ***
            Financi

In [1]:
"""
Figure 1: Net Sector-Specific HHI Effects on Forward Risk Reversal (Model 3A)
"""

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from matplotlib.patches import Patch

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.8,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "figure.dpi": 150,
})

BASELINE = -0.0859  # M3A HHI baseline coefficient (omitted reference: Communication)

# (sector, interaction_coef, clustered_t, DK6_t)
sector_data = [
    ("Real Estate",            0.2976,  3.52,  0.95),
    ("Health Care",            0.1627,  3.18,  2.29),
    ("Utilities",              0.1377,  1.35,  0.67),
    ("Materials",              0.1246,  4.29,  2.18),
    ("Industrials",            0.1096,  5.68,  2.54),
    ("Energy",                 0.1074,  4.49,  2.94),
    ("Consumer Staples",       0.1007,  3.90,  1.90),
    ("Consumer Discretionary", 0.0592,  3.20,  1.60),
    ("Financials",             0.0388,  0.51,  0.23),
    ("Technology",             0.0131,  0.51,  0.24),
    ("Communication",          0.0000,  None,  None),  # reference, baseline only
]

sectors, net_effects, clust_sig, dk_sig = [], [], [], []
for name, interact, t_c, t_dk in sector_data:
    sectors.append(name)
    net_effects.append(BASELINE + interact)
    if name == "Communication":
        clust_sig.append(None)
        dk_sig.append(None)
    else:
        clust_sig.append(abs(t_c) >= 1.96)
        dk_sig.append(abs(t_dk) >= 1.96)

# Sort by net effect, descending
order = np.argsort(net_effects)[::-1]
sectors = [sectors[i] for i in order]
net_effects = [net_effects[i] for i in order]
clust_sig = [clust_sig[i] for i in order]
dk_sig = [dk_sig[i] for i in order]

# Color palette
COLOR_POS_DK    = "#1F4E79"
COLOR_POS_CLUST = "#5B9BD5"
COLOR_POS_INSIG = "#A9C5E8"
COLOR_NEG_DK    = "#8C2A2A"
COLOR_NEG_CLUST = "#C66060"
COLOR_NEG_INSIG = "#E0B0B0"
COLOR_REF       = "#7F7F7F"

bar_colors = []
for i, name in enumerate(sectors):
    if name == "Communication":
        bar_colors.append(COLOR_REF)
    elif net_effects[i] > 0:
        if dk_sig[i]:        bar_colors.append(COLOR_POS_DK)
        elif clust_sig[i]:   bar_colors.append(COLOR_POS_CLUST)
        else:                bar_colors.append(COLOR_POS_INSIG)
    else:
        if dk_sig[i]:        bar_colors.append(COLOR_NEG_DK)
        elif clust_sig[i]:   bar_colors.append(COLOR_NEG_CLUST)
        else:                bar_colors.append(COLOR_NEG_INSIG)

fig, ax = plt.subplots(figsize=(7.2, 5.2))

ypos = np.arange(len(sectors))
ax.barh(ypos, net_effects, color=bar_colors,
        edgecolor="black", linewidth=0.6, height=0.72)
ax.axvline(0, color="black", linewidth=0.8, zorder=0)

ax.set_yticks(ypos)
ax.set_yticklabels(sectors)
ax.invert_yaxis()
ax.set_xlabel(r"Net effect on $RR_{j,t+1}$ (baseline $+$ sector interaction)")

# Value labels with significance markers
for i, val in enumerate(net_effects):
    name = sectors[i]
    if name == "Communication":
        text = f"{val:+.3f} (ref.)"
        ax.text(val + 0.005, i, text, va="center", ha="left",
                fontsize=8.5, color="0.25")
    else:
        if dk_sig[i] and clust_sig[i]:
            marker = r"$^{\dagger\ddagger}$"
        elif clust_sig[i]:
            marker = r"$^{\dagger}$"
        else:
            marker = ""
        offset = 0.005 if val >= 0 else -0.005
        ha = "left" if val >= 0 else "right"
        ax.text(val + offset, i, f"{val:+.3f}{marker}",
                va="center", ha=ha, fontsize=8.5)

xmin = min(net_effects) - 0.05
xmax = max(net_effects) + 0.07
ax.set_xlim(xmin, xmax)
ax.grid(axis="x", linestyle=":", linewidth=0.5, alpha=0.6, zorder=0)

legend_handles = [
    Patch(facecolor=COLOR_POS_DK,    edgecolor="black", linewidth=0.6,
          label=r"Positive, robust DK 5%"),
    Patch(facecolor=COLOR_POS_CLUST, edgecolor="black", linewidth=0.6,
          label=r"Positive, clustered 5%"),
    Patch(facecolor=COLOR_POS_INSIG, edgecolor="black", linewidth=0.6,
          label=r"Positive, insig."),
    Patch(facecolor=COLOR_NEG_CLUST, edgecolor="black", linewidth=0.6,
          label=r"Negative, clustered 5%"),
    Patch(facecolor=COLOR_NEG_INSIG, edgecolor="black", linewidth=0.6,
          label=r"Negative, insig."),
    Patch(facecolor=COLOR_REF,       edgecolor="black", linewidth=0.6,
          label="Reference (Comm.)"),
]
ax.legend(handles=legend_handles, loc="lower right",
          frameon=True, framealpha=0.95, edgecolor="0.7",
          fontsize=8, handlelength=1.2, handleheight=0.9,
          ncol=2, columnspacing=0.9)

plt.tight_layout()

plt.savefig("./Figures/fig_net_sector_effects.pdf", bbox_inches="tight")
plt.savefig("./Figures/fig_net_sector_effects.png", bbox_inches="tight", dpi=200)
plt.close()

In [3]:
"""
Figure 2 (revised): Supplier HHI Evolution by Sector

Small-multiples view of inherited upstream concentration over the full
sample for all 11 GICS sectors. The figure motivates Supplier HHI as a
meaningful, time-varying, and cross-sectionally heterogeneous variable
without making a comovement claim that monthly data cannot support
visually.

Sectors are arranged in a 4x3 grid sorted by mean Supplier HHI (highest
first, so the most-concentrated upstream profiles read top-left). All
panels share a common y-axis range so cross-sector comparisons are
meaningful at a glance.

INPUTS:
    Master_Predictive_Panel.csv  (set PANEL_PATH below)

OUTPUTS:
    fig_supplier_hhi_evolution.pdf
    fig_supplier_hhi_evolution.png

USAGE:
    python fig_supplier_hhi_evolution.py
"""

import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
PANEL_PATH = "Master_Predictive_Panel.csv"

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 0.7,
    "figure.dpi": 150,
})

# Highlight color for the upstream-channel sectors (those for which the
# M3B Supplier HHI sector interaction is significant under entity
# clustering). The remaining seven sectors are drawn in a subdued gray
# so the eye is drawn to the four that carry the regression evidence.
UPSTREAM_CHANNEL = {"Real Estate", "Financials", "Health Care", "Industrials"}
COLOR_HIGHLIGHT  = "#1F4E79"   # deep navy
COLOR_MUTED      = "#7F7F7F"   # gray

# -----------------------------------------------------------------------------
# Load panel
# -----------------------------------------------------------------------------
if not os.path.exists(PANEL_PATH):
    print(f"ERROR: panel file not found at {PANEL_PATH!r}.")
    print("Edit PANEL_PATH at the top of this script to point at your master panel.")
    sys.exit(1)

panel = pd.read_csv(PANEL_PATH)
panel["YearMonth"] = pd.to_datetime(panel["YearMonth"])
panel = panel.sort_values(["Sector", "YearMonth"])

required_cols = {"Sector", "YearMonth", "Supplier_HHI"}
missing = required_cols - set(panel.columns)
if missing:
    print(f"ERROR: panel missing required columns {missing}")
    sys.exit(1)

# -----------------------------------------------------------------------------
# Sector ordering: by mean Supplier HHI, descending
# -----------------------------------------------------------------------------
mean_supp = panel.groupby("Sector")["Supplier_HHI"].mean().sort_values(ascending=False)
sectors = list(mean_supp.index)

# Compute global y-range so all panels are directly comparable.
# A small headroom is added so labels and ticks don't crash into the spines.
ymin = max(0.0, panel["Supplier_HHI"].min() * 0.95)
ymax = panel["Supplier_HHI"].max() * 1.05

# -----------------------------------------------------------------------------
# Build 4x3 small multiples (12 slots; the 12th slot stays blank for legend)
# -----------------------------------------------------------------------------
n_rows, n_cols = 4, 3
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10.5, 9.2),
                         sharex=True, sharey=False)
axes_flat = axes.flatten()

# Determine common x range from all data so empty XLRE/XLC pre-launch
# periods show up as visible gaps rather than rescaled axes
xmin = panel["YearMonth"].min()
xmax = panel["YearMonth"].max()

for i, sector in enumerate(sectors):
    ax = axes_flat[i]
    sub = panel[panel["Sector"] == sector].dropna(subset=["Supplier_HHI"])

    color = COLOR_HIGHLIGHT if sector in UPSTREAM_CHANNEL else COLOR_MUTED
    lw    = 1.3              if sector in UPSTREAM_CHANNEL else 1.0
    alpha = 1.0              if sector in UPSTREAM_CHANNEL else 0.85

    ax.plot(sub["YearMonth"], sub["Supplier_HHI"],
            color=color, linewidth=lw, alpha=alpha)

    # Annotate panel with sector name and the channel marker
    title_marker = r"$^{\dagger}$" if sector in UPSTREAM_CHANNEL else ""
    ax.set_title(f"{sector}{title_marker}", loc="left",
                 fontweight="bold" if sector in UPSTREAM_CHANNEL else "normal",
                 fontsize=10)

    # Light reference line at the sector's full-sample mean
    sector_mean = sub["Supplier_HHI"].mean()
    ax.axhline(sector_mean, color=color, linewidth=0.5,
               linestyle="--", alpha=0.4)

    # Sample mean as a small text annotation in the upper-right corner
    ax.text(0.97, 0.93, f"mean = {sector_mean:.3f}",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=7.5, color="0.3",
            bbox=dict(boxstyle="round,pad=0.2",
                      facecolor="white", edgecolor="none", alpha=0.7))

    ax.set_xlim(xmin, xmax)
    # ax.set_ylim(ymin, ymax)
    ax.grid(axis="y", linestyle=":", linewidth=0.4, alpha=0.5)

    # Axis labels only on the outer left/bottom panels (sharex/sharey
    # already strip interior tick labels, but we still want the axis-name
    # text to be sparse)
    row, col = divmod(i, n_cols)
    if col == 0:
        ax.set_ylabel("Supplier HHI")
    if row == n_rows - 1 or i >= len(sectors) - n_cols:
        ax.set_xlabel("Date")

# Hide the unused 12th panel and use it for a legend / explanatory block
for j in range(len(sectors), n_rows * n_cols):
    axes_flat[j].axis("off")

# Legend / explanation in the unused 12th panel
legend_ax = axes_flat[-1]
legend_ax.axis("off")

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color=COLOR_HIGHLIGHT, linewidth=1.6,
           label=r"Upstream-channel sector ($\dagger$)"),
    Line2D([0], [0], color=COLOR_MUTED, linewidth=1.2,
           label="Other sector"),
    Line2D([0], [0], color="0.3", linewidth=0.6,
           linestyle="--", label="Sector full-sample mean"),
]
legend_ax.legend(handles=legend_handles, loc="upper left",
                 frameon=False, fontsize=8.5,
                 handlelength=1.6, handleheight=1.0,
                 labelspacing=0.7)

plt.tight_layout()

plt.savefig("./Figures/fig_supplier_hhi_evolution.pdf", bbox_inches="tight")
plt.savefig("./Figures/fig_supplier_hhi_evolution.png", bbox_inches="tight", dpi=200)
print("Saved: fig_supplier_hhi_evolution.pdf")
print("Saved: fig_supplier_hhi_evolution.png")
plt.close()

Saved: fig_supplier_hhi_evolution.pdf
Saved: fig_supplier_hhi_evolution.png


In [5]:
"""
Figure 3: Inter-Sector Input-Output Weight Matrix Heatmap

Visualizes the omega matrix derived from the BEA Make/Use tables, where
omega[i, j] is the share of sector j's inputs sourced from sector i.
The diagonal is zero by construction (a sector cannot inherit network
fragility from itself), so the heatmap shows the full off-diagonal
structure of inter-sector dependence in the U.S. economy.

This figure provides the visual complement to the Supplier HHI variable:
columns with mass concentrated on a few rows correspond to sectors with
high upstream concentration. The reader can see at a glance which
downstream sectors depend on which upstream hubs.

INPUT OPTIONS (the script tries each in order; first match wins):
    1. A pickled dict of {year: omega_DataFrame} at OMEGA_PICKLE_PATH
       (this is what your data_cleaning script computes as `annual_weights`)
    2. A single CSV with rows = input sector, cols = output sector,
       at OMEGA_CSV_PATH

If you have neither, save your annual_weights dict as a pickle:
    import pickle
    with open("annual_weights.pkl", "wb") as f:
        pickle.dump(annual_weights, f)

USAGE:
    python fig_io_heatmap.py

One small caption tweak worth considering: add a sentence noting that this is 
the matrix used (with a one-year lag) for the most recent panel observations, 
which connects the figure to the regression more directly.
"""

import os
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap

# -----------------------------------------------------------------------------
# Configuration: edit these paths
# -----------------------------------------------------------------------------
OMEGA_PICKLE_PATH = "./Data/annual_weights.pkl"
OMEGA_CSV_PATH    = "./Data/omega_matrix_2024.csv"  # fallback if pickle absent
TARGET_YEAR       = 2024

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi": 150,
})

# Logical sector ordering (groups with similar upstream dependencies near each other)
SECTOR_ORDER = [
    "Real Estate", "Financials",
    "Health Care", "Consumer Staples", "Consumer Discretionary",
    "Industrials", "Materials", "Energy", "Utilities",
    "Technology", "Communication",
]

# -----------------------------------------------------------------------------
# Load omega matrix
# -----------------------------------------------------------------------------
omega = None
if os.path.exists(OMEGA_PICKLE_PATH):
    print(f"Loading omega dict from {OMEGA_PICKLE_PATH}")
    with open(OMEGA_PICKLE_PATH, "rb") as f:
        annual_weights = pickle.load(f)
    available = sorted(annual_weights.keys())
    yr = TARGET_YEAR if TARGET_YEAR in available else available[-1]
    omega = annual_weights[yr].copy()
    used_year = yr
elif os.path.exists(OMEGA_CSV_PATH):
    print(f"Loading omega matrix from {OMEGA_CSV_PATH}")
    omega = pd.read_csv(OMEGA_CSV_PATH, index_col=0)
    used_year = TARGET_YEAR
else:
    print("ERROR: neither annual_weights.pkl nor omega_matrix.csv found.")
    print("Save your annual_weights dict from data_cleaning.py as a pickle:")
    print('  with open("annual_weights.pkl", "wb") as f:')
    print('      pickle.dump(annual_weights, f)')
    sys.exit(1)

# Reorder to logical sector ordering, dropping anything not in SECTOR_ORDER
present = [s for s in SECTOR_ORDER if s in omega.index and s in omega.columns]
omega = omega.loc[present, present].copy()
omega = omega.astype(float)

# Sanity check: diagonal should be zero
diag = np.diag(omega.values)
if np.any(np.abs(diag) > 1e-12):
    print(f"WARNING: diagonal not strictly zero (max abs = {np.abs(diag).max():.4f}); zeroing it now.")
    np.fill_diagonal(omega.values, 0)

# -----------------------------------------------------------------------------
# Heatmap
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7.5, 6.5))

# Custom sequential colormap: white -> deep navy
cmap = LinearSegmentedColormap.from_list(
    "navy_seq", ["#FFFFFF", "#D6E1EE", "#5B9BD5", "#1F4E79"], N=256
)

# Cap the color scale at the 95th percentile of off-diagonal values so a
# small number of very large flows don't wash out the rest of the structure
off_diag = omega.values[~np.eye(len(omega), dtype=bool)]
vmax = max(np.percentile(off_diag, 95), 0.05)

im = ax.imshow(omega.values, cmap=cmap, vmin=0, vmax=vmax, aspect="equal")

# Tick labels
ax.set_xticks(np.arange(len(omega)))
ax.set_yticks(np.arange(len(omega)))
ax.set_xticklabels(omega.columns, rotation=40, ha="right")
ax.set_yticklabels(omega.index)

ax.set_xlabel("Downstream sector $j$ (consumer)", labelpad=8)
ax.set_ylabel("Upstream sector $k$ (supplier)", labelpad=8)

# Annotate cells with the value (omit zeros and very small values)
for i in range(len(omega)):
    for j in range(len(omega)):
        v = omega.values[i, j]
        if i == j:
            ax.text(j, i, "—", ha="center", va="center",
                    color="0.5", fontsize=8)
        elif v >= 0.01:
            text_color = "white" if v > vmax * 0.5 else "0.15"
            ax.text(j, i, f"{v:.2f}".lstrip("0"),
                    ha="center", va="center",
                    color=text_color, fontsize=7.5)

# Colorbar
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label("Input share (off-diagonal)", rotation=270, labelpad=15)

plt.tight_layout()

plt.savefig("./Figures/fig_io_heatmap.pdf", bbox_inches="tight")
plt.savefig("./Figures/fig_io_heatmap.png", bbox_inches="tight", dpi=200)
print(f"Saved: fig_io_heatmap.pdf  (year {used_year})")
print(f"Saved: fig_io_heatmap.png  (year {used_year})")
plt.close()

Loading omega dict from ./Data/annual_weights.pkl
Saved: fig_io_heatmap.pdf  (year 2024)
Saved: fig_io_heatmap.png  (year 2024)


---

## Old Code

In [6]:
# 1. Load Data
df = pd.read_csv('Master_Predictive_Panel.csv')

# 2. Format Dates
df['YearMonth'] = pd.to_datetime(df['YearMonth'])

# 3. Create the t+1 Forward-Looking Dependent Variable
# We shift the 25-Delta Risk Reversal backward so row 't' predicts 't+1' tail pricing
df.sort_values(by=['Ticker', 'YearMonth'], inplace=True)
df['Fwd_RR25'] = df.groupby('Ticker')['Risk_Reversal_25'].shift(-1)

# 4. Set the MultiIndex required for PanelOLS
df.set_index(['Ticker', 'YearMonth'], inplace=True)

# 5. Handle NaNs cleanly
df_clean = df.dropna(subset=['Fwd_RR25', 'Risk_Reversal_50', 'tot_return_index_gross_dvds']).copy()

# ==========================================
# IN-SAMPLE INFERENCE (MODELS 1, 2, & 3)
# ==========================================

# Define controls: We add the ATM 50-delta spread as a baseline control
controls = ['Risk_Reversal_50', 'tot_return_index_gross_dvds']

# -------------------------------------------------------------------
# MODEL 1: The Baseline Structural Model
# Equation: RR25_t+1 = HHI_t + SupplierHHI_t + Controls + SectorFE + TimeFE
# -------------------------------------------------------------------
exog1 = sm.add_constant(df_clean[['HHI', 'Supplier_HHI'] + controls])
mod1 = PanelOLS(df_clean['Fwd_RR25'], exog1, entity_effects=True, time_effects=True)
res1 = mod1.fit(cov_type='clustered', cluster_entity=True)

# -------------------------------------------------------------------
# MODEL 2: The Bayesian Interaction Model
# Equation: RR25_t+1 = HHI_t * Cascade_t + Controls + SectorFE + TimeFE
# -------------------------------------------------------------------
df_clean['HHI_x_Cascade'] = df_clean['HHI'] * df_clean['Cascade_Trigger']

exog2 = sm.add_constant(df_clean[['HHI', 'Cascade_Trigger', 'HHI_x_Cascade'] + controls])
mod2 = PanelOLS(df_clean['Fwd_RR25'], exog2, entity_effects=True, time_effects=True)
res2 = mod2.fit(cov_type='clustered', cluster_entity=True)

# -------------------------------------------------------------------
# MODEL 3: Sector Heterogeneity 
# Equation: RR25_t+1 = HHI_t * D_Sector + Controls + SectorFE + TimeFE
# -------------------------------------------------------------------
sector_dummies = pd.get_dummies(df_clean['Sector'], drop_first=True, dtype=float)
interact_cols = []

for col in sector_dummies.columns:
    interact_name = f'HHI_x_{col}'
    df_clean[interact_name] = df_clean['HHI'] * sector_dummies[col]
    interact_cols.append(interact_name)

exog3 = sm.add_constant(df_clean[['HHI'] + interact_cols + controls])
mod3 = PanelOLS(df_clean['Fwd_RR25'], exog3, entity_effects=True, time_effects=True)
res3 = mod3.fit(cov_type='clustered', cluster_entity=True)

# Print Summary Comparison Table
print(compare({"M1 Structural": res1, "M2 Bayesian": res2, "M3 Heterogeneity": res3}))

# ==========================================
# OUT-OF-SAMPLE (OOS) FORECASTING
# ==========================================

# Reset index to easily filter by date
df_oos = df_clean.reset_index()

# Define Training (2005-2019) and Testing (2020-Present) Windows
train = df_oos[(df_oos['YearMonth'] >= '2005-01-01') & (df_oos['YearMonth'] <= '2019-12-31')].set_index(['Ticker', 'YearMonth'])
test = df_oos[df_oos['YearMonth'] >= '2025-01-01'].set_index(['Ticker', 'YearMonth'])

# Variables for forecasting: Including Time-Series Macro Controls
forecast_controls = ['Risk_Reversal_50', 'tot_return_index_gross_dvds', 'VIX', 'Term_Spread', 'Credit_Spread']

# 1. Baseline Model (AR + Macro Controls)
exog_base_train = sm.add_constant(train[forecast_controls])
mod_base = PanelOLS(train['Fwd_RR25'], exog_base_train, entity_effects=True, time_effects=False)
res_base = mod_base.fit()

exog_base_test = sm.add_constant(test[forecast_controls])
pred_base = res_base.predict(exog_base_test)
mspe_base = mean_squared_error(test['Fwd_RR25'], pred_base)

# 2. Challenger Model (Baseline + HHI + SupplierHHI)
exog_chal_train = sm.add_constant(train[['HHI', 'Supplier_HHI'] + forecast_controls])
mod_chal = PanelOLS(train['Fwd_RR25'], exog_chal_train, entity_effects=True, time_effects=False)
res_chal = mod_chal.fit()

exog_chal_test = sm.add_constant(test[['HHI', 'Supplier_HHI'] + forecast_controls])
pred_chal = res_chal.predict(exog_chal_test)
mspe_chal = mean_squared_error(test['Fwd_RR25'], pred_chal)

print("\n--- Out-of-Sample Forecasting Performance ---")
print(f"Baseline OOS MSPE: {mspe_base:.6f}")
print(f"HHI Challenger OOS MSPE: {mspe_chal:.6f}")
print(f"Reduction in Error: {((mspe_base - mspe_chal) / mspe_base) * 100:.2f}%")

                               Model Comparison                              
                                 M1 Structural   M2 Bayesian M3 Heterogeneity
-----------------------------------------------------------------------------
Dep. Variable                         Fwd_RR25      Fwd_RR25         Fwd_RR25
Estimator                             PanelOLS      PanelOLS         PanelOLS
No. Observations                          2425          2425             2425
Cov. Est.                            Clustered     Clustered        Clustered
R-squared                               0.0108        0.0066           0.0114
R-Squared (Within)                     -0.0314        0.0021           0.0111
R-Squared (Between)                    -2.1932        0.1022          -30.717
R-Squared (Overall)                    -0.0481        0.0033          -0.1741
F-statistic                             5.8992        2.8756           1.9036
P-value (F-stat)                        0.0001        0.0136    

In [11]:
import pandas as pd
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, compare

# 1. Setup Base Variables and Controls (Assuming df_clean is already set with MultiIndex)
controls = ['Risk_Reversal_50', 'tot_return_index_gross_dvds']
sector_dummies = pd.get_dummies(df_clean['Sector'], drop_first=True, dtype=float)

# 2. Re-create interaction terms for internal HHI
hhi_interact_cols = []
for col in sector_dummies.columns:
    interact_name = f'HHI_x_{col}'
    df_clean[interact_name] = df_clean['HHI'] * sector_dummies[col]
    hhi_interact_cols.append(interact_name)

# 3. Create NEW interaction terms for upstream Supplier_HHI
supp_interact_cols = []
for col in sector_dummies.columns:
    interact_name = f'SuppHHI_x_{col}'
    df_clean[interact_name] = df_clean['Supplier_HHI'] * sector_dummies[col]
    supp_interact_cols.append(interact_name)

# ==========================================
# REVISED MODELS
# ==========================================

# Model 2 (Revised): Cascade as an additive control, NO interaction
exog2_rev = sm.add_constant(df_clean[['HHI', 'Cascade_Trigger'] + controls])
mod2_rev = PanelOLS(df_clean['Fwd_RR25'], exog2_rev, entity_effects=True, time_effects=True)
res2_rev = mod2_rev.fit(cov_type='clustered', cluster_entity=True)

# Model 3 (Revised): Heterogeneous HHI + Homogeneous SupplierHHI
exog3_rev = sm.add_constant(df_clean[['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + controls])
mod3_rev = PanelOLS(df_clean['Fwd_RR25'], exog3_rev, entity_effects=True, time_effects=True)
res3_rev = mod3_rev.fit(cov_type='clustered', cluster_entity=True)

# Path A: The "Kitchen Sink" (Heterogeneous HHI AND Heterogeneous SupplierHHI)
exog_path_a = sm.add_constant(df_clean[['HHI'] + hhi_interact_cols + ['Supplier_HHI'] + supp_interact_cols + controls])
mod_path_a = PanelOLS(df_clean['Fwd_RR25'], exog_path_a, entity_effects=True, time_effects=True)
res_path_a = mod_path_a.fit(cov_type='clustered', cluster_entity=True)

# Path B: The "Alternating" Model (Homogeneous HHI + Heterogeneous SupplierHHI)
exog_path_b = sm.add_constant(df_clean[['HHI', 'Supplier_HHI'] + supp_interact_cols + controls])
mod_path_b = PanelOLS(df_clean['Fwd_RR25'], exog_path_b, entity_effects=True, time_effects=True)
res_path_b = mod_path_b.fit(cov_type='clustered', cluster_entity=True)

# ==========================================
# PRINT RESULTS
# ==========================================
print(compare({"M2 (No Cascade Interact)": res2_rev, 
               "M3 (+ Homogeneous SuppHHI)": res3_rev, 
               "Path A (Kitchen Sink)": res_path_a, 
               "Path B (Alternating)": res_path_b}))

                                                          Model Comparison                                                         
                                     M2 (No Cascade Interact) M3 (+ Homogeneous SuppHHI) Path A (Kitchen Sink) Path B (Alternating)
-----------------------------------------------------------------------------------------------------------------------------------
Dep. Variable                                        Fwd_RR25                   Fwd_RR25              Fwd_RR25             Fwd_RR25
Estimator                                            PanelOLS                   PanelOLS              PanelOLS             PanelOLS
No. Observations                                         2425                       2425                  2425                 2425
Cov. Est.                                           Clustered                  Clustered             Clustered            Clustered
R-squared                                              0.0065               